# Anisotropic Friction Mathematical Formulation

We scale tangential velocity element-wise (`τ_aniso = μ_aniso ⊙ τ`), use the existing `smooth_mu()` pipeline with `||τ_aniso||`, and get simple derivatives `∂τ_aniso/∂τ = diag(μ_aniso)`. Default `μ_aniso = (1,1)` is isotropic.

**Notation:** τ = tangential velocity (tangent basis), μ_aniso = scaling vector, τ_aniso = μ_aniso ⊙ τ, ε_v = velocity threshold, f₀/f₁ = mollifiers, μ(y) = smooth static→kinetic coefficient.

### Anisotropic Scaling

Notation: `τ` = tangential relative velocity (tangent basis), `μ_aniso` = scaling vector (default `(1,1)`). Scaled velocity (element-wise product):

```
τ_aniso = μ_aniso ⊙ τ   →   τ_aniso[i] = μ_aniso[i] * τ[i]
```

### Friction Force

Same isotropic formulation with `τ_aniso` in place of `τ`:

```
F = -μ N f₁(||τ_aniso||)/||τ_aniso|| T τ_aniso
```

`μ` from `smooth_mu(||τ_aniso||, μ_s, μ_k)`; `N` = normal force magnitude, `T` = tangent basis, `f₁` = mollifier.


In [ ]:
from sympy import *
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, HTML
import plotly
plotly.offline.init_notebook_mode()
display(HTML(
    '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
))


In [ ]:
# Symbolic variables
x = Symbol('x', real=True)
eps_v = Symbol(r'\epsilon_v', real=True, positive=True)
mu_s, mu_k = symbols(r'\mu_s \mu_k', real=True, positive=True)
mu_x, mu_y = symbols(r'\mu_x \mu_y', real=True, positive=True)

In [ ]:
import re

def plot(*fs, title=None, min_x=-2, max_x=2, even=False, odd=False, **subs):
    subs = {eps_v: subs.get("eps_v", 1e-3), 
            mu_s: subs.get("mu_s", 1), 
            mu_k: subs.get("mu_k", 0.1),
            mu_x: subs.get("mu_x", 1),
            mu_y: subs.get("mu_y", 1)}
    xs = np.linspace(min_x*subs[eps_v], max_x*subs[eps_v], 201)
    s = np.sign(xs) if odd else np.ones_like(xs)

    def ys(f):
        result = []
        for xi in xs:
            x_val = abs(xi) if (even or odd) else xi
            try:
                val = f.subs({x: x_val} | subs)
                # Handle Piecewise and other complex expressions
                if hasattr(val, 'evalf'):
                    val = val.evalf()
                result.append(float(val))
            except:
                result.append(0.0)
        return s * np.array(result, dtype=float)

    fig = go.Figure(
        [go.Scatter(x=xs, y=ys(f)) for f in fs],
        layout=dict(
            width=800, height=600, template="plotly_dark",
            xaxis_title=r'speed', title=title
        ))
    fig.show()

def print_latex(expr):
    print(latex(expr).replace(r"\\", r"\\" + "\n"))

def print_code(expr):
    print(cxxcode(expr).replace(r"\epsilon", "eps").replace(r"\mu", "mu").replace("x", "y"))


In [ ]:
def f0(x):
    """Smooth friction mollifier"""
    return x * x * (1 - x / (3 * eps_v)) / eps_v + eps_v / 3

sym_f0 = Piecewise(
    (x, x >= eps_v),
    (f0(x), x < eps_v)
)

display(Eq(Symbol("f_{0}(x)"), sym_f0.expand()))
print_latex(sym_f0.expand())

plot(sym_f0, title=r'$f_0(x)$', even=True)

In [ ]:
def f1(x):
    """Derivative of f0"""
    return x * (2 - x / eps_v) / eps_v

sym_f1 = Piecewise(
    (f1(x), x <= eps_v),
    (1, x > eps_v)
)

display(Eq(Symbol("f_{1}(x)"), sym_f1.expand()))

plot(sym_f1, title=r"$f_1(x)$", odd=True)

In [ ]:
sym_f2 = sym_f1.diff(x)

display(Eq(Symbol("f_{2}(x)"), sym_f2.simplify()))

plot(sym_f2, title=r"$f_2(x)$", even=True)


## Smooth Coefficient of Friction

Piecewise quadratic transition from static $\mu_s$ to kinetic $\mu_k$.


In [ ]:
def mu_quadratic_part1(x):
    return (2 / eps_v**2) * (mu_k - mu_s) * x**2 + mu_s

def mu_quadratic_part2(x):
    return -2 * (mu_k - mu_s) / eps_v**2 * (x - eps_v)**2 + mu_k

sym_mu = Piecewise(
    (mu_quadratic_part1(x), x <= eps_v / 2),
    (mu_quadratic_part2(x), x <= eps_v),
    (mu_k, True)
).simplify()

display(Eq(Symbol(r"\mu(x)"), sym_mu))

print_latex(sym_mu)

plot(sym_mu, title=r"$\mu(x)$", even=True, mu_s=0.5, mu_k=0.3)
plot(sym_mu.diff(x), title=r"$\mu'(x)$", odd=True, mu_s=0.5, mu_k=0.3)

## Combined Friction Function

The combined friction function is $\mu(x) \cdot f_1(x)$, which represents the friction force magnitude per unit normal force.


In [ ]:
sym_mu_f1 = (sym_mu * sym_f1).simplify()

display(Eq(Symbol(r"\mu(x) \cdot f_1(x)"), sym_mu_f1))

plot(sym_mu_f1, title=r"$\mu(x) \cdot f_1(x)$", odd=True, mu_s=0.5, mu_k=0.3)


## Normalized Friction Function

$\frac{\mu(x) \cdot f_1(x)}{x}$ in the force expression:

$$F = -\frac{\mu(\|\tau\|) \cdot f_1(\|\tau\|)}{\|\tau\|} \cdot N \cdot \tau$$


In [ ]:
sym_mu_f1_over_x = (sym_mu_f1 / x).simplify()

display(Eq(Symbol(r"\frac{\mu(x) \cdot f_1(x)}{x}"), sym_mu_f1_over_x))

plot(sym_mu_f1_over_x, title=r"$\frac{\mu(x) \cdot f_1(x)}{x}$", even=True, mu_s=0.5, mu_k=0.3)


## Anisotropic Scaling (symbolic)

Same as above: $\tau_{\text{aniso}} = \mu_{\text{aniso}} \odot \tau$ (element-wise). Notation: $\tau \in \mathbb{R}^2$ tangential velocity, $\mu_{\text{aniso}} \in \mathbb{R}^2$ scaling (default $(1,1)$).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

mu_aniso = np.array([2.0, 0.5])
theta = np.linspace(0, 2*np.pi, 16)
tau = np.column_stack([np.cos(theta), np.sin(theta)])
tau_aniso = tau * mu_aniso
idx = 2

fig, ax = plt.subplots(figsize=(10, 10), facecolor='white')
ax.set_aspect('equal')
ax.quiver(np.zeros(16), np.zeros(16), tau[:, 0], tau[:, 1], 
          color='blue', alpha=0.5, width=0.003, label='τ')
ax.quiver(np.zeros(16), np.zeros(16), tau_aniso[:, 0], tau_aniso[:, 1], 
          color='red', alpha=0.7, width=0.003, label='τ_aniso')
ax.add_patch(plt.Circle((0, 0), 1, fill=False, color='blue', linestyle='--', linewidth=1.5))
ax.add_patch(mpatches.Ellipse((0, 0), 2*mu_aniso[0], 2*mu_aniso[1], 
                              fill=False, color='red', linestyle='--', linewidth=1.5))
ax.quiver(0, 0, tau[idx, 0], tau[idx, 1], color='darkblue', width=0.005, linewidth=2)
ax.quiver(0, 0, tau_aniso[idx, 0], tau_aniso[idx, 1], color='darkred', width=0.005, linewidth=2)
ax.annotate(f'τ=[{tau[idx,0]:.2f},{tau[idx,1]:.2f}]', xy=tau[idx]*1.1, fontsize=10, color='darkblue')
ax.annotate(f'τ_aniso=[{tau_aniso[idx,0]:.2f},{tau_aniso[idx,1]:.2f}]', 
            xy=tau_aniso[idx]*1.1, fontsize=10, color='darkred')
ax.set(xlim=(-3, 3), ylim=(-3, 3), xlabel='Tangent axis 0', ylabel='Tangent axis 1', 
       title=f'Anisotropic Scaling: μ_aniso = {mu_aniso}')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## Effect on Friction Coefficient

Scaling shifts when the static–kinetic transition occurs. Friction coefficient is evaluated at $\|\tau_{\text{aniso}}\|$: $\mu = \mu(\|\mu_{\text{aniso}} \odot \tau\|)$.


In [ ]:
# Symbolic anisotropic magnitude
tau_x, tau_y = symbols('tau_x tau_y', real=True)
tau_aniso_mag = sqrt((mu_x * tau_x)**2 + (mu_y * tau_y)**2)

display(Eq(Symbol(r"\|\tau_{aniso}\|"), tau_aniso_mag))

# For a unit vector in direction theta
theta_sym = Symbol('theta', real=True)
tau_unit = Matrix([cos(theta_sym), sin(theta_sym)])
tau_aniso_unit = Matrix([mu_x * cos(theta_sym), mu_y * sin(theta_sym)])
tau_aniso_mag_theta = sqrt((mu_x * cos(theta_sym))**2 + (mu_y * sin(theta_sym))**2)

display(Eq(Symbol(r"\|\tau_{aniso}(\theta)\|"), tau_aniso_mag_theta.simplify()))


In [ ]:
# Numeric evaluation: effect on smooth_mu transition
mu_s_val, mu_k_val, eps_v_val = 0.5, 0.3, 0.001
tau_mags = np.linspace(0, 2*eps_v_val, 200)

# Different anisotropy scenarios
scenarios = [
    ("Isotropic [1,1]", [1.0, 1.0]),
    ("Aniso 2x [2,1]", [2.0, 1.0]),
    ("Aniso 0.5x [0.5,1]", [0.5, 1.0]),
    ("Aniso 2x [2,2]", [2.0, 2.0])
]

traces = []
for name, mu_a in scenarios:
    # Evaluate at direction [1,0] for simplicity
    scale = np.linalg.norm(np.array(mu_a) * [1, 0])
    tau_aniso_mags = tau_mags * scale
    
    # Evaluate smooth_mu at anisotropic magnitudes
    mu_vals = []
    for t in tau_aniso_mags:
        try:
            val = sym_mu.subs({x: t, eps_v: eps_v_val, mu_s: mu_s_val, mu_k: mu_k_val})
            # Handle Piecewise and other complex expressions
            if hasattr(val, 'evalf'):
                val = val.evalf()
            mu_vals.append(float(val))
        except:
            mu_vals.append(0.0)
    
    traces.append(go.Scatter(x=tau_mags, y=mu_vals, name=name))

go.Figure(data=traces, 
          layout=dict(width=800, height=600, template="plotly_dark",
                      xaxis_title=r'$\|\tau\|$ (original magnitude)',
                      yaxis_title=r'$\mu(\|\tau_{aniso}\|)$',
                      title='Effect of Anisotropy on Friction Coefficient')).show()


## Friction Force Computation

$$F = -\frac{\mu(\|\tau_{\text{aniso}}\|) \cdot f_1(\|\tau_{\text{aniso}}\|)}{\|\tau_{\text{aniso}}\|} N \cdot T \cdot \tau_{\text{aniso}}$$

$N$ = normal force; $\mu$ from `smooth_mu(||τ_aniso||, μ_s, μ_k)`; $T$ = tangent basis; $\tau_{\text{aniso}} = \mu_{\text{aniso}} \odot \tau$.


In [ ]:
# Visualize friction force magnitude for different anisotropies
mu_s_val, mu_k_val, eps_v_val = 0.5, 0.3, 0.001
tau_mags = np.linspace(0, 2*eps_v_val, 200)

# Different anisotropy scenarios
scenarios = [
    ("Isotropic [1,1]", [1.0, 1.0]),
    ("Aniso 2x [2,1]", [2.0, 1.0]),
    ("Aniso 0.5x [0.5,1]", [0.5, 1.0]),
    ("Aniso 2x [2,2]", [2.0, 2.0])
]

traces = []
for name, mu_a in scenarios:
    scale = np.linalg.norm(np.array(mu_a) * [1, 0])
    tau_aniso_mags = tau_mags * scale
    
    # Evaluate mu * f1 / x at anisotropic magnitudes
    force_coeffs = []
    for t in tau_aniso_mags:
        if t > 1e-10:
            try:
                val = sym_mu_f1_over_x.subs({
                    x: t, eps_v: eps_v_val, mu_s: mu_s_val, mu_k: mu_k_val
                })
                # Handle Piecewise and other complex expressions
                if hasattr(val, 'evalf'):
                    val = val.evalf()
                val = float(val)
            except:
                val = 0.0
            force_coeffs.append(val * t)  # Multiply back by magnitude for force
        else:
            force_coeffs.append(0.0)
    
    traces.append(go.Scatter(x=tau_mags, y=force_coeffs, name=name))

go.Figure(data=traces,
          layout=dict(width=800, height=600, template="plotly_dark",
                      xaxis_title=r'$\|\tau\|$ (original magnitude)',
                      yaxis_title='Friction force (per unit N)',
                      title='Friction Force: Isotropic vs Anisotropic')).show()


## Derivative Computation

$$\frac{\partial \tau_{\text{aniso}}}{\partial \tau} = \text{diag}(\mu_{\text{aniso}}) \quad \text{(constant diagonal matrix)}$$

### Chain Rule for Force Jacobian

$$\frac{\partial F}{\partial v} = \frac{\partial F}{\partial \tau_{\text{aniso}}} \cdot \frac{\partial \tau_{\text{aniso}}}{\partial \tau} \cdot \frac{\partial \tau}{\partial v}$$

Steps: compute `∂F/∂τ_aniso` (isotropic formulation), then multiply by `diag(μ_aniso)` and by `Tᵀ`.


In [ ]:
def smooth_mu(y, mu_s, mu_k, eps_v):
    if abs(y) >= eps_v: return mu_k
    z = abs(y) / eps_v
    return (2*(mu_k-mu_s)*z*z + mu_s) if abs(y) < 0.5*eps_v else (-2*(mu_k-mu_s)*(z*(z-2)+1) + mu_k)

def f1(y, eps_v):
    return y*(2-y/eps_v)/eps_v if y <= eps_v and y > 0 else (1.0 if y > eps_v else 0.0)

def mu_f1(y, mu_s, mu_k, eps_v):
    return smooth_mu(y, mu_s, mu_k, eps_v) * f1(y, eps_v)

def mu_f0(y, mu_s, mu_k, eps_v):
    if y <= 0: return 0.0
    t_vals = np.linspace(0, min(y, eps_v), max(100, int(y*1000)))
    if len(t_vals) < 2: return 0.0
    f1_vals = [mu_f1(t, mu_s, mu_k, eps_v) for t in t_vals]
    integral = np.trapz(f1_vals, t_vals)
    if y > eps_v:
        integral += mu_k * (y - eps_v)
    return integral

def mu_f2(y, mu_s, mu_k, eps_v):
    if y <= 0 or y > eps_v: return 0.0
    mu_val = smooth_mu(y, mu_s, mu_k, eps_v)
    f1_val = f1(y, eps_v)
    dmu_dy = (4*(mu_k-mu_s)*y/eps_v**2) if y < 0.5*eps_v else (-4*(mu_k-mu_s)*(y-eps_v)/eps_v**2)
    df1_dy = (2/eps_v - 2*y/eps_v**2) if y <= eps_v else 0.0
    return dmu_dy * f1_val + mu_val * df1_dy

mu_s, mu_k, eps_v = 0.5, 0.3, 0.001
ys = np.linspace(0.0001, 2*eps_v, 200, dtype=np.float64)

go.Figure(data=[
    go.Scatter(x=ys, y=[mu_f0(y, mu_s, mu_k, eps_v) for y in ys], name="f0"),
    go.Scatter(x=ys, y=[mu_f1(y, mu_s, mu_k, eps_v) for y in ys], name="f1"),
    go.Scatter(x=ys, y=[mu_f2(y, mu_s, mu_k, eps_v) for y in ys], name="f2"),
    go.Scatter(x=ys, y=[mu_f1(y, mu_s, mu_k, eps_v)/y if y > 0 else 0.0 for y in ys], name="f1/x"),
    go.Scatter(x=ys, y=[(mu_f2(y, mu_s, mu_k, eps_v)*y - mu_f1(y, mu_s, mu_k, eps_v))/y**3 if y > 0 else 0.0 for y in ys], name="f2_x-f1/x³"),
], layout=dict(width=800, height=600, template="simple_white")).show()


In [ ]:
def smooth_mu(y, mu_s, mu_k, eps_v):
    if mu_s == mu_k or abs(y) >= eps_v: return mu_k
    z = abs(y) / eps_v
    return (2*(mu_k-mu_s)*z*z + mu_s) if abs(y) < 0.5*eps_v else (-2*(mu_k-mu_s)*(z*(z-2)+1) + mu_k)

def smooth_mu_derivative(y, mu_s, mu_k, eps_v):
    if mu_s == mu_k or abs(y) >= eps_v: return 0.0
    z = abs(y) / eps_v
    return (4*(mu_k-mu_s)*z/eps_v) if abs(y) < 0.5*eps_v else (-4*(mu_k-mu_s)*(z-1)/eps_v)

mu_s, mu_k, eps_v = 0.5, 0.3, 0.001
ys = np.linspace(0, 3*eps_v, 300, dtype=np.float64)
go.Figure(data=[
    go.Scatter(x=ys, y=[smooth_mu(y, mu_s, mu_k, eps_v) for y in ys], name="μ"),
    go.Scatter(x=ys, y=[smooth_mu_derivative(y, mu_s, mu_k, eps_v) for y in ys], name="μ'"),
    go.Scatter(x=[eps_v, eps_v], y=[0, mu_k], mode='lines', line=dict(dash='dash', color='gray'), name='ε_v'),
], layout=dict(width=800, height=600, template="simple_white", 
               xaxis_title=r'$||\tau_{aniso}||$', yaxis_title='μ',
               title='smooth_mu: Static to Kinetic Transition')).show()

In [ ]:
def smooth_mu(y, mu_s, mu_k, eps_v):
    if abs(y) >= eps_v: return mu_k
    z = abs(y) / eps_v
    return (2*(mu_k-mu_s)*z*z + mu_s) if abs(y) < 0.5*eps_v else (-2*(mu_k-mu_s)*(z*(z-2)+1) + mu_k)

mu_s, mu_k, eps_v = 0.5, 0.3, 0.001
ys = np.linspace(0, 2*eps_v, 200, dtype=np.float64)
scenarios = [("Isotropic [1,1]", 1.0), ("Aniso 2x [2,1]", 2.0), ("Aniso 0.5x [0.5,1]", 0.5)]

go.Figure(data=[go.Scatter(x=ys, y=[smooth_mu(y*s, mu_s, mu_k, eps_v) for y in ys], name=n) 
                for n, s in scenarios],
          layout=dict(width=800, height=600, template="simple_white",
                      xaxis_title=r'$||\tau||$', yaxis_title='μ',
                      title='Effect of Anisotropy on smooth_mu')).show()


In [ ]:
from sympy import *
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

x_sym, eps_v_sym = symbols('x \\epsilon_v', real=True, positive=True)

f0 = x_sym**2 * (1 - x_sym/(3*eps_v_sym)) / eps_v_sym + eps_v_sym/3
f1 = x_sym * (2 - x_sym/eps_v_sym) / eps_v_sym

sym_f0 = Piecewise((x_sym, x_sym >= eps_v_sym), (f0, x_sym < eps_v_sym))
sym_f1 = Piecewise((f1, x_sym <= eps_v_sym), (1, x_sym > eps_v_sym))
sym_f2 = sym_f1.diff(x_sym)
sym_f1_over_x = sym_f1 / x_sym
sym_f2_x_minus_f1_over_x3 = (sym_f2 * x_sym - sym_f1) / x_sym**3

display(Eq(Symbol("f_0"), sym_f0.expand()))
display(Eq(Symbol("f_1"), sym_f1.expand()))
display(Eq(Symbol("f_2"), sym_f2.simplify()))

eps_v_val = 0.001
xs = np.linspace(0.0001*eps_v_val, 3*eps_v_val, 201)
subs = {eps_v_sym: eps_v_val}

def eval_sym(f, xi):
    try:
        return float(f.subs({x_sym: xi} | subs)) if xi > 0 else 0.0
    except:
        return 0.0

go.Figure(data=[
    go.Scatter(x=xs, y=[eval_sym(sym_f0, xi) for xi in xs], name="f0"),
    go.Scatter(x=xs, y=[eval_sym(sym_f1, xi) for xi in xs], name="f1"),
    go.Scatter(x=xs, y=[eval_sym(sym_f2, xi) for xi in xs], name="f2"),
    go.Scatter(x=xs, y=[eval_sym(sym_f1_over_x, xi) for xi in xs], name="f1/x"),
    go.Scatter(x=xs, y=[eval_sym(sym_f2_x_minus_f1_over_x3, xi) for xi in xs], name="f2_x-f1/x³"),
], layout=dict(width=800, height=600, template="plotly_dark",
              xaxis_title=r'$||\tau_{aniso}||$', title='Friction Mollifier Functions')).show()


In [ ]:
def smooth_mu(y, mu_s, mu_k, eps_v):
    if abs(y) >= eps_v: return mu_k
    z = abs(y) / eps_v
    return (2*(mu_k-mu_s)*z*z + mu_s) if abs(y) < 0.5*eps_v else (-2*(mu_k-mu_s)*(z*(z-2)+1) + mu_k)

def f1_over_x(y, eps_v):
    if y <= eps_v: return (2 - y/eps_v) / eps_v if y > 0 else 0.0
    return 1.0 / y if y > 0 else 0.0

mu_s, mu_k, eps_v = 0.5, 0.3, 0.001
tau_mags = np.linspace(0, 2*eps_v, 200, dtype=np.float64)
scenarios = [("Isotropic [1,1]", [1,1]), ("Aniso 2x [2,1]", [2,1]), ("Aniso 0.5x [0.5,1]", [0.5,1]), ("Aniso 2x [2,2]", [2,2])]

traces = []
for name, mu_a in scenarios:
    scale = np.linalg.norm(np.array(mu_a) * [1,0])
    tau_aniso = tau_mags * scale
    forces = [smooth_mu(t, mu_s, mu_k, eps_v) * f1_over_x(t, eps_v) * t if t > 1e-10 else 0.0 for t in tau_aniso]
    traces.append(go.Scatter(x=tau_mags, y=forces, name=name))

go.Figure(data=traces, layout=dict(width=800, height=600, template="simple_white",
                                   xaxis_title=r'$||\tau||$', yaxis_title='Force',
                                   title='Force: Isotropic vs Anisotropic')).show()


In [ ]:
mu_aniso = np.array([2.0, 0.5])
T = np.array([[1.0, 0.5], [0.5, 1.0], [0.0, 0.0]])
dF_dtau_aniso = np.array([0.1, 0.2])
dF_dv = dF_dtau_aniso @ np.diag(mu_aniso) @ T.T
print(f"μ_aniso={mu_aniso}, diag(μ_aniso)=\n{np.diag(mu_aniso)}\n∂F/∂v={dF_dv}")


## Integration with smooth_mu Pipeline

Use `||τ_aniso||` instead of `||τ||` and pass it to `smooth_mu(||τ_aniso||, μ_s, μ_k)`. Static when `||τ_aniso|| < ε_v` (μ_s), kinetic when ≥ ε_v (μ_k); `smooth_mu()` interpolates in between. Anisotropy only changes when the transition happens, not the mechanism.


In [ ]:
def smooth_mu_example(y, mu_s, mu_k, eps_v=0.001):
    if abs(y) >= eps_v: return mu_k
    z = abs(y) / eps_v
    return (2*(mu_k-mu_s)*z*z + mu_s) if abs(y) < 0.5*eps_v else (-2*(mu_k-mu_s)*(z*(z-2)+1) + mu_k)

tau_iso = np.array([0.0005, 0.0005])
mu_aniso = np.array([2.0, 0.5])
tau_aniso = mu_aniso * tau_iso
norm_iso, norm_aniso = np.linalg.norm(tau_iso), np.linalg.norm(tau_aniso)
print(f"||τ||={norm_iso:.6f}, μ={smooth_mu_example(norm_iso, 0.5, 0.3):.4f} → ||τ_aniso||={norm_aniso:.6f}, μ={smooth_mu_example(norm_aniso, 0.5, 0.3):.4f}")
